# Optimal Control for Spacecraft Landing


In [ ]:
from pathlib import Path
import csv
import json
import math
import subprocess
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

Q32_DIR = Path("q3.2_results")
Q33_DIR = Path("q3.3_results")
Q4_DIR = Path("q4_results")

from lander.params import P
from lander.lcvx import LanderSOCP, is_lossless, solve_min_fuel, solve_min_landing_error
from lander.pointmass import B1_CASES, B1_UNREACHABLE, simulate, glide_violation, run_b1


## Question 3.2 — Case B


In [ ]:
def case_b(
    x0=(-6.0, 9.0, -6.0, 0.0), fuel_kg=0.17, N=40,
    output_dir=Q32_DIR,
):
    x0 = np.asarray(x0, dtype=float)
    m0 = P.m_dry + float(fuel_kg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    center = LanderSOCP(P, N=N)
    relaxed, _ = solve_min_fuel(center, x0, m0, certify=False)
    print("Pad center reachable in relaxed search:", relaxed is not None)
    assert relaxed is None, "A relaxed center solution was found"

    sol, d_star, evaluations = solve_min_landing_error(P, x0, m0, N=N)
    if sol is None:
        raise RuntimeError(f"No certified two-stage solution; Stage 1 d* = {d_star}")
    sim = simulate(sol, P, mode="accel")

    report = "\n".join([
        f"Stage 1 minimum miss distance: {d_star:.9f} m",
        f"Flight time: {sol.tgo:.9f} s",
        f"Stage 2 miss distance: {sol.landing_error:.9f} m",
        f"Fuel used: {1000*sol.fuel:.6f} g",
        f"Fuel remaining: {1000*(fuel_kg-sol.fuel):.6f} g",
        f"Lossless: {is_lossless(sol)}",
        f"Maximum relative gap: {np.max(sol.relaxation_gap/np.maximum(sol.sigma,1e-9)):.9g}",
        f"Simulated position error from planned touchdown: {sim['final_pos_err']:.9g} m",
        f"Simulated terminal speed: {sim['final_vel_err']:.9g} m/s",
        f"Thrust violation: {sim['thrust_violation']:.9g} N",
        f"Glide-slope violation: {glide_violation(sim['traj'], 1.):.9g} m",
    ])
    print(report)
    (output_dir / "summary.txt").write_text(report + "\n", encoding="utf-8")
    np.savetxt(output_dir / "time_search.csv", evaluations, delimiter=",",
               header="tf,landing_error", comments="")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(sol.p[:, 0], sol.p[:, 1], "o-", markersize=2, label="SOCP trajectory")
    ax.plot(sim["traj"][:, 0], sim["traj"][:, 1], "--", label="Simulation")
    xx = np.linspace(min(sol.p[:, 0].min(), sol.p[-1, 0])-2,
                     max(0.0, sol.p[:, 0].max())+1, 300)
    ax.plot(xx, np.abs(xx-sol.p[-1, 0]), ":", label="Glide slope")
    ax.scatter([0, sol.p[-1, 0]], [0, 0], label="Pad / touchdown")
    ax.set(xlabel="Horizontal position [m]", ylabel="Altitude [m]",
           title="Q3.2 - Case B: minimum landing error")
    ax.set_aspect("equal")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "trajectory.png", dpi=200)
    display(fig)
    plt.close(fig)

    vals = np.asarray(evaluations, dtype=float)
    fig, ax = plt.subplots(figsize=(8, 4))
    mask = np.isfinite(vals[:, 1])
    ax.plot(vals[mask, 0], vals[mask, 1], "o-")
    ax.scatter(sol.tgo, d_star, label="Best evaluated time")
    ax.set(xlabel="Flight time [s]", ylabel="Miss distance [m]",
           title="Q3.2 - Minimum miss distance vs. flight time")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "error_vs_time.png", dpi=200)
    display(fig)
    plt.close(fig)
    return {"solution": sol, "d_star": d_star, "evaluations": evaluations,
            "simulation": sim, "report": report}

In [ ]:
q32_result = case_b()


## Question 3.3 — Benchmark B1


In [ ]:
N_DEFAULT = 40
GRID_DEFAULT = 25
TOL_DEFAULT = 0.02
OUTPUT_COLUMNS = (
    "case", "reachable", "method", "ok", "landed", "tgo_s", "fuel_g",
    "landing_error_m", "d_star_m", "gap_max_mps2", "gap_relative_max",
    "lossless", "thrust_violation_N", "glide_violation_m", "solve_time_s",
)


In [ ]:
def state_key(x0, m0):
    return tuple(float(x) for x in np.r_[x0, m0])


def to_float(value):
    try:
        x = float(value)
        return x if np.isfinite(x) else None
    except (TypeError, ValueError):
        return None


class Guidance:

    def __init__(self, certify: bool, N=N_DEFAULT, n_grid=GRID_DEFAULT, tol=TOL_DEFAULT):
        self.certify, self.N, self.n_grid, self.tol = certify, N, n_grid, tol
        self.fuel_socp = LanderSOCP(P, N=N)
        self.cache = {}

    def two_stage(self, x0, m0):
        return solve_min_landing_error(
            P, x0, m0, N=self.N, n_grid=self.n_grid,
            tol=self.tol, certify=self.certify,
        )

    def guidance(self, x0, m0):
        key = state_key(x0, m0)
        if key not in self.cache:
            sol, _ = solve_min_fuel(
                self.fuel_socp, x0, m0, n_grid=self.n_grid,
                tol=self.tol, certify=self.certify,
            )
            d_star, method = None, 'min_fuel'
            if sol is None:
                sol, d_star, _ = self.two_stage(x0, m0)
                method = 'min_landing_error'
            self.cache[key] = dict(sol=sol, d_star=d_star, method=method)
        return self.cache[key]['sol']

    def get_d_star(self, x0, m0):
        key = state_key(x0, m0)
        if key not in self.cache:
            self.guidance(x0, m0)
        entry = self.cache[key]
        if entry['d_star'] is None:
            _sol, d_star, _ = self.two_stage(x0, m0)
            entry['d_star'] = d_star
        value = to_float(entry['d_star'])
        return value if value is not None else 0.0


def test_b1(certify, N=N_DEFAULT, n_grid=GRID_DEFAULT, tol=TOL_DEFAULT):
    planner = Guidance(certify=certify, N=N, n_grid=n_grid, tol=tol)
    t_start = time.perf_counter()
    official = run_b1(planner.guidance, planner.get_d_star, p=P)
    elapsed = time.perf_counter() - t_start
    table, sols, rel_gaps = [], {}, np.full((len(B1_CASES), N), np.nan)

    for i, row in enumerate(official):
        case = B1_CASES[i]
        x0, m0 = case[:4], P.m_dry + case[4]
        entry = planner.cache.get(state_key(x0, m0), {})
        sol = entry.get('sol')
        sols[i] = sol
        if sol is None:
            max_gap = relative = None
            lossless = False
        else:
            gap = np.maximum(sol.relaxation_gap, 0.0)
            relative_gaps = gap / np.maximum(sol.sigma, 1e-9)
            rel_gaps[i, :] = relative_gaps
            max_gap = float(np.max(gap))
            relative = float(np.max(relative_gaps))
            lossless = bool(is_lossless(sol))

        table.append({
            'case': i,
            'reachable': bool(i not in B1_UNREACHABLE),
            'method': entry.get('method', 'no_solution'),
            'ok': bool(row.get('ok', False)),
            'landed': bool(row.get('landed', False)),
            'tgo_s': to_float(sol.tgo if sol is not None else None),
            'fuel_g': to_float(1000.0 * row['fuel']) if 'fuel' in row else None,
            'landing_error_m': to_float(row.get('d_land')),
            'd_star_m': to_float(row.get('d_star')),
            'gap_max_mps2': max_gap,
            'gap_relative_max': relative,
            'lossless': lossless,
            'thrust_violation_N': to_float(row.get('viol_T')),
            'glide_violation_m': to_float(row.get('viol_gs')),
            'solve_time_s': float(row['t_solve']),
        })
    return dict(certify=certify, rows=table, solutions=sols,
                relative_gaps=rel_gaps, elapsed_s=elapsed)


def summary_b1(result):
    rows = result['rows']
    valid = [r for r in rows if r['ok']]
    def maximum(key):
        vals = [r[key] for r in valid if r[key] is not None]
        return max(vals) if vals else None
    def mean(key):
        vals = [r[key] for r in valid if r[key] is not None]
        return float(np.mean(vals)) if vals else None
    return dict(
        certified_search=result['certify'],
        cases=len(rows), solved=len(valid), landed=sum(r['landed'] for r in rows),
        reachable_landed=sum(r['landed'] for r in rows if r['reachable']),
        unreachable_landed=sum(r['landed'] for r in rows if not r['reachable']),
        plans_lossy=sum(not r['lossless'] for r in valid),
        mean_fuel_g=mean('fuel_g'), total_fuel_g=sum(r['fuel_g'] for r in valid),
        max_gap_mps2=maximum('gap_max_mps2'),
        max_relative_gap=maximum('gap_relative_max'),
        max_thrust_violation_N=maximum('thrust_violation_N'),
        max_glide_violation_m=maximum('glide_violation_m'),
        mean_guidance_time_s=float(np.mean([r['solve_time_s'] for r in rows])),
        max_guidance_time_s=float(np.max([r['solve_time_s'] for r in rows])),
        total_run_time_s=result['elapsed_s'],
    )


def save_csv(path, rows):
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


def table_b1(rows):
    header = '| Case | Method | Landed | Fuel (g) | Miss (m) | Max gap (m/s²) | Viol. thrust (N) | Viol. glide (m) | Guidance (s) |'
    separator = '|---:|---|:---:|---:|---:|---:|---:|---:|---:|'
    lines = [header, separator]
    def num(x, digits):
        return '—' if x is None else f'{x:.{digits}f}'
    for r in rows:
        lines.append('| {case} | {method} | {landed} | {fuel} | {miss} | {gap} | {T} | {gs} | {sec} |'.format(
            case=r['case'], method=('fuel' if r['method'] == 'min_fuel' else 'error'),
            landed='Yes' if r['landed'] else 'No',
            fuel=num(r['fuel_g'], 2), miss=num(r['landing_error_m'], 3),
            gap=num(r['gap_max_mps2'], 5), T=num(r['thrust_violation_N'], 3),
            gs=num(r['glide_violation_m'], 4), sec=num(r['solve_time_s'], 3),
        ))
    return '\n'.join(lines)


In [ ]:
def plot_b1(uncertified, certified, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.3), sharex=True, sharey=True,
                             constrained_layout=True)
    all_finite = np.r_[uncertified['relative_gaps'].ravel(),
                       certified['relative_gaps'].ravel()]
    all_finite = all_finite[np.isfinite(all_finite)]
    common_vmax = max(0.01, float(np.max(all_finite)) if all_finite.size else 0.01)
    for ax, res, title in zip(axes, (uncertified, certified),
                              ('certify=False', 'certify=True')):
        heat = np.ma.masked_invalid(res['relative_gaps'])
        cmap = plt.get_cmap('viridis').copy()
        cmap.set_bad(color='#dddddd')
        mesh = ax.imshow(heat, origin='lower', aspect='auto', interpolation='none',
                         cmap=cmap, vmin=0.0, vmax=common_vmax)
        ax.set_title(title)
        ax.set_xlabel('Time-step k')
        ax.set_yticks(np.arange(len(B1_CASES)))
        ax.set_ylabel('B1 case index')
        ax.axhline(16.5, color='white', ls='--', lw=1.0)
        fig.colorbar(mesh, ax=ax, shrink=0.78, label='max(0, σ − ||u||) / σ')
    fig.suptitle('Q3.3: nodal relaxation gaps across all B1 cases')
    fig.savefig(out_dir / 'b1_nodal_relaxation_gaps.png', dpi=180)
    display(fig)
    plt.close(fig)

    lossy = [(r['gap_relative_max'], r['case']) for r in uncertified['rows']
             if r['ok'] and r['gap_relative_max'] is not None and r['gap_relative_max'] > 1e-4]
    if not lossy:
        return None
    _, case_index = max(lossy)
    sol = uncertified['solutions'][case_index]
    sim = simulate(sol, P, mode='accel')
    t_sub = np.linspace(0, sol.tgo, len(sim['thrust']), endpoint=False)
    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, constrained_layout=True)
    ax = axes[0]
    ax.plot(t_sub, sim['thrust'], lw=1.5, label='Simulated physical thrust ||T(t)||')
    ax.axhline(P.rho1, color='red', ls='--', label='Minimum allowed thrust ρ₁')
    ax.axhline(P.rho2, color='gray', ls='--', label='Maximum allowed thrust ρ₂')
    ax.set_ylabel('Thrust (N)')
    ax.legend(loc='best', fontsize=8)
    ax = axes[1]
    ax.step(sol.t[:-1], np.maximum(sol.relaxation_gap, 0.0), where='post',
            label='Relaxation gap σₖ − ||uₖ||')
    ax.axhline(0.0, color='gray', lw=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Gap (m/s²)')
    ax.legend(loc='best', fontsize=8)
    fig.suptitle(f'B1 case {case_index}: an uncertified plan can demand impossible thrust')
    fig.savefig(out_dir / 'b1_lossy_case_thrust.png', dpi=180)
    display(fig)
    plt.close(fig)
    return case_index


def report_b1(uncertified, certified, example):
    a, b = summary_b1(uncertified), summary_b1(certified)
    def fmt(value, dp=3):
        return 'n/a' if value is None else f'{value:.{dp}f}'
    lines = [
        '# Q3.3 — Benchmark B1 (20 frozen initial states)',
        '',
        '## Comparison of certification modes',
        '',
        '| Metric | certify=False | certify=True |',
        '|---|---:|---:|',
        f'| Landed / 20 | {a["landed"]}/20 | {b["landed"]}/20 |',
        f'| Reachable cases landed / 17 | {a["reachable_landed"]}/17 | {b["reachable_landed"]}/17 |',
        f'| Unreachable cases landed / 3 | {a["unreachable_landed"]}/3 | {b["unreachable_landed"]}/3 |',
        f'| Solved plans | {a["solved"]} | {b["solved"]} |',
        f'| Non-lossless plans (relative gap > 1e-4) | {a["plans_lossy"]} | {b["plans_lossy"]} |',
        f'| Mean simulated fuel, solved cases (g) | {fmt(a["mean_fuel_g"],2)} | {fmt(b["mean_fuel_g"],2)} |',
        f'| Max thrust-bound violation (N) | {fmt(a["max_thrust_violation_N"],3)} | {fmt(b["max_thrust_violation_N"],3)} |',
        f'| Max glide-slope violation (m) | {fmt(a["max_glide_violation_m"],4)} | {fmt(b["max_glide_violation_m"],4)} |',
        f'| Mean guidance time (s/case) | {fmt(a["mean_guidance_time_s"],3)} | {fmt(b["mean_guidance_time_s"],3)} |',
        f'| Max guidance time (s/case) | {fmt(a["max_guidance_time_s"],3)} | {fmt(b["max_guidance_time_s"],3)} |',
        f'| Total wall-clock evaluation (s) | {fmt(a["total_run_time_s"],2)} | {fmt(b["total_run_time_s"],2)} |',
        '',
    ]
    lines += [
        '## Uncertified search', '', table_b1(uncertified['rows']), '',
        '## Certified search', '', table_b1(certified['rows']), '',
    ]
    return '\n'.join(lines)


In [ ]:
def benchmark_b1(output_dir=Q33_DIR, N=N_DEFAULT, n_grid=GRID_DEFAULT,
                      tol=TOL_DEFAULT):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    runs = []
    for certify in (False, True):
        res = test_b1(certify, N=N, n_grid=n_grid, tol=tol)
        runs.append(res)
        label = "certified" if certify else "uncertified"
        save_csv(output_dir / f"b1_{label}.csv", res["rows"])

    example = plot_b1(runs[0], runs[1], output_dir)
    report = report_b1(runs[0], runs[1], example)
    (output_dir / "q33_report.md").write_text(report, encoding="utf-8")
    display(Markdown(report))
    return {"uncertified": runs[0], "certified": runs[1], "report": report}

In [ ]:
q33_result = benchmark_b1()


## Question 4 — Benchmark B2


In [ ]:
Q4_POLICIES = {
    "LCvx cascade": "lander.controllers:make_cascade",
    "Heuristic": "submissions/baseline_heuristic.py",
    "PPO": "submissions/baseline_ppo.py",
}
Q4_CONDITIONS = ("nominal", "heavy", "light", "gusty", "storm")


def percent(x):
    return f"{100*x:.1f}%"


def num(x, digits=2):
    return f"{x:.{digits}f}" if math.isfinite(x) else "n/a"


def benchmark_b2(n=100, processes=2, output_dir=Q4_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    summaries = {}

    for name, source in Q4_POLICIES.items():
        slug = name.lower().replace(" ", "_")
        result_file = output_dir / f"b2_{slug}_n{n}.json"
        cmd = [
            sys.executable, "-m", "lander.evaluate", "--policy", source,
            "--benchmark", "b2", "--n", str(n), "--processes", str(processes),
            "--out", str(result_file),
        ]
        print(f"\n=== {name}: {n} seeds/condition ===", flush=True)
        subprocess.run(cmd, check=True)
        summaries[name] = json.loads(result_file.read_text(encoding="utf-8"))["summary"]

    lines = [
        f"# Q4 - Benchmark B2 ({n} seeds per condition)", "",
        "Fuel is mean seconds of full main-engine thrust among successful episodes.",
        "Time is mean controller computation time per step, in milliseconds.",
        "", "## Overall", "",
        "| Policy | Success | Mean score | Fuel (s, successes) | Time (ms/step) |",
        "|---|---:|---:|---:|---:|",
    ]
    for name, s in summaries.items():
        a = s["overall"]
        lines.append(f"| {name} | {percent(a['success'])} | {num(a['score'])} | "
                     f"{num(a['fuel_success'])} | {num(a['policy_ms'])} |")

    lines.extend([
        "", "## By environment", "",
        "| Environment | Policy | Success | Fuel (s, successes) | Time (ms/step) | Off-pad | Crash/hard |",
        "|---|---|---:|---:|---:|---:|---:|",
    ])
    for scenario in Q4_CONDITIONS:
        for name, s in summaries.items():
            a = s["per_scenario"][scenario]
            lines.append(f"| {scenario} | {name} | {percent(a['success'])} | "
                         f"{num(a['fuel_success'])} | {num(a['policy_ms'])} | "
                         f"{percent(a['off_pad'])} | {percent(a['crash']+a['hard'])} |")
    report = "\n".join(lines)
    report_file = output_dir / f"Q4_B2_comparison_n{n}.md"
    report_file.write_text(report, encoding="utf-8")
    display(Markdown(report))
    return {"summaries": summaries, "report": report, "report_file": report_file}


In [ ]:
q4_result = benchmark_b2(n=100, processes=2)


Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/usr/local/Caskroom/miniconda/base/envs/lander/lib/python3.12/multiprocessing/pool.py", line 131, in worker
    put((job, i, result))
  File "/usr/local/Caskroom/miniconda/base/envs/lander/lib/python3.12/multiprocessing/queues.py", line 399, in put
    self._writer.send_bytes(obj)
  File "/usr/local/Caskroom/miniconda/base/envs/lander/lib/python3.12/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
  File "/usr/local/Caskroom/miniconda/base/envs/lander/lib/python3.12/multiprocessing/connection.py", line 427, in _send_bytes
    self._send(header + buf)
  File "/usr/local/Caskroom/miniconda/base/envs/lander/lib/python3.12/multiprocessing/connection.py", line 384, in _send
    n = write(self._handle, buf)
        ^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent 